In [1]:
from screenshot_scanner import ScreenshotScanner
import os


In [2]:
folder_path = "/home/zuzia/Desktop/studia/year2/creativity/photo_book/photos"
# Initialize scanner
scanner = ScreenshotScanner()

# Process an image (simple one-liner)
result = scanner.process("/home/zuzia/Desktop/studia/year2/creativity/photo_book/photos/20161001_120119.jpg")

print(result)

{'is_screenshot': False, 'score': 4, 'confidence': 40.0, 'reasons': ['Common aspect ratio: 1.33', 'Low ELA: 0.88', 'High horizontal edges: 0.54', 'Low noise: 6.32']}


In [3]:
# #I want to check all the images in the folder 
# def check_screenshots_in_folder(folder_path):
#     # Initialize scanner
#     scanner = ScreenshotScanner()

#     # List to store results
#     results = []

#     # Iterate through all files in the folder
#     for filename in os.listdir(folder_path):
#         file_path = os.path.join(folder_path, filename)
        
#         # Check if it's a file (not a directory)
#         if os.path.isfile(file_path):
#             # Process the image and append the result
#             result = scanner.process(file_path)
#             results.append((filename, result))

#         print(f"Processed {filename}: {result}")

#     return results

In [4]:
#check_results = check_screenshots_in_folder(folder_path)

**Not working well need to found something other**

In [3]:
import cv2
import numpy as np


def is_screenshot_by_ui(image_path):
  """Detects screenshots by looking for top status bars and bottom navigation bars

  using edge and contour analysis.
  """
  image = cv2.imread(image_path)
  if image is None:
    return False

  height, width, _ = image.shape

  # 1. Check the top 7% (Status Bar area) and bottom 7% (Navigation bar area)
  top_strip = image[0 : int(height * 0.07), 0:width]
  bottom_strip = image[int(height * 0.93) : height, 0:width]

  for strip in [top_strip, bottom_strip]:
    gray = cv2.cvtColor(strip, cv2.COLOR_BGR2GRAY)

    # Apply Canny edge detection to find crisp UI lines and icon shapes
    edges = cv2.Canny(gray, 50, 150)

    # Find horizontal lines or distinct clustered bounding boxes (common for status bar icons)
    contours, _ = cv2.findContours(
        edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    # Count small sharp elements typical of wifi/battery icons or clock text
    ui_elements_count = 0
    for c in contours:
      x, y, w, h = cv2.boundingRect(c)
      # Status bar icons are typically small and clustered near the top/bottom edges
      if 5 < w < 60 and 5 < h < 30:
        ui_elements_count += 1

    # If we find multiple small UI-like icons packed into the top/bottom edge strip
    if ui_elements_count >= 4:
      return True

  return False

In [4]:
from PIL import Image


def is_screenshot(image_path):
  # Check 1: Does it have camera hardware EXIF data?
  try:
    with Image.open(image_path) as img:
      exif = img._getexif()
      if exif:
        # If it has a camera Make/Model tag, it's 100% a real photo, skip UI checks
        for tag_id, value in exif.items():
          if tag_id in [271, 272]:  # 271: Make, 272: Model
            return False
  except Exception:
    pass

  # Check 2: If no camera EXIF exists, run the UI status bar check
  return is_screenshot_by_ui(image_path)

In [5]:
#make a folder for non-screenshots
import shutil
non_screenshot_folder = os.path.join(folder_path, "non_screenshots")
if not os.path.exists(non_screenshot_folder):
    os.makedirs(non_screenshot_folder)

In [8]:


for filename in os.listdir(folder_path):
    image_path = os.path.join(folder_path, filename)
    if not os.path.isfile(image_path):
        continue
    screenshot_check = is_screenshot(image_path)
    print(f"File: {filename}, Screenshot Check: {screenshot_check}")
    #if it not a screenshot, copy it to the non_screenshots folder
    
    if not screenshot_check:
        destination_path = os.path.join(non_screenshot_folder, filename)
        shutil.copy2(image_path, destination_path)
        print(f"Copied {filename} to non_screenshots folder.")


File: 20161001_115547.jpg, Screenshot Check: False
Copied 20161001_115547.jpg to non_screenshots folder.
File: FB_IMG_1497814227299.jpg, Screenshot Check: True
File: 20161001_121545.jpg, Screenshot Check: False
Copied 20161001_121545.jpg to non_screenshots folder.
File: bad_quality.jpg, Screenshot Check: False
Copied bad_quality.jpg to non_screenshots folder.
File: Screenshot_2017-06-24-21-05-19.png, Screenshot Check: True
File: 20161001_121535.jpg, Screenshot Check: False
Copied 20161001_121535.jpg to non_screenshots folder.
File: 20161001_115640.jpg, Screenshot Check: False
Copied 20161001_115640.jpg to non_screenshots folder.
File: 20170225_193528.jpg, Screenshot Check: False
Copied 20170225_193528.jpg to non_screenshots folder.
File: 20161001_115620.jpg, Screenshot Check: False
Copied 20161001_115620.jpg to non_screenshots folder.
File: 20161001_121659.jpg, Screenshot Check: False
Copied 20161001_121659.jpg to non_screenshots folder.
File: panda.png, Screenshot Check: True
File: 20

In [8]:
#Do the function from the cell avove
def check_screenshots_in_folder(folder_path):
    for filename in os.listdir(folder_path):
        image_path = os.path.join(folder_path, filename)
        if not os.path.isfile(image_path):
            continue
        screenshot_check = is_screenshot(image_path)
        #print(f"File: {filename}, Screenshot Check: {screenshot_check}")
        #if it not a screenshot, copy it to the non_screenshots folder
        
        if not screenshot_check:
            destination_path = os.path.join(non_screenshot_folder, filename)
            shutil.copy2(image_path, destination_path)
            #print(f"Copied {filename} to non_screenshots folder.")
    print("Finished non screenshot check and copying.")

In [9]:
check_screenshots_in_folder(folder_path)

Finished non screenshot check and copying.
